In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AnalisisModeloLATAM")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.sql.shuffle.partitions", "48")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 21:24:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.4
Master: yarn


In [2]:
ruta_integrado = "hdfs:///proyecto_latam/processed/integrado"

df = spark.read.parquet(ruta_integrado)

print("Total de vuelos:", df.count())
print("Total de columnas:", len(df.columns))

df.groupBy("delay_target").count().orderBy("delay_target").show()

Total de vuelos: 1149291
Total de columnas: 83


[Stage 4:============================================>              (3 + 1) / 4]

+------------+------+
|delay_target| count|
+------------+------+
|           0|861553|
|           1|287738|
+------------+------+



In [3]:
from pyspark.sql.functions import avg, round

df.select(
    round(avg("DepDelayMinutes"), 2).alias("promedio_demora_minutos"),
    round(avg("delay_target") * 100, 2).alias("porcentaje_demora_15_min")
).show()

[Stage 7:============================================>              (3 + 1) / 4]

+-----------------------+------------------------+
|promedio_demora_minutos|porcentaje_demora_15_min|
+-----------------------+------------------------+
|                  17.84|                   25.04|
+-----------------------+------------------------+



In [4]:
ruta_metricas = (
    "hdfs:///proyecto_latam/outputs/metricas/modelo_definitivo"
)

metricas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_metricas)
)

metricas.orderBy("f1", ascending=False).show(
    20,
    truncate=False
)

+-------------------+------+------------------+------------------+------+-----+-----+-----+-------------------+-------------------+-------------------+-------+------------------+
|modelo             |umbral|roc_auc           |pr_auc            |tn    |fp   |fn   |tp   |precision          |recall             |f1                 |alertas|alertas_por_1000  |
+-------------------+------+------------------+------------------+------+-----+-----+-----+-------------------+-------------------+-------------------+-------+------------------+
|logistic_regression|0.4   |0.7108754060962722|0.528721779911512 |71234 |47948|16452|39769|0.4533784785161371 |0.7073691325305491 |0.5525851408245216 |87717  |500.088367929853  |
|random_forest      |0.4   |0.7114629627943252|0.5175282633466612|77182 |42000|18838|37383|0.4709194663844904 |0.6649294747514274 |0.5513554172443291 |79383  |452.5749274527802 |
|gbt_classifier     |0.4   |0.7082917438572005|0.5112151432134311|73703 |45479|17564|38657|0.459458495768

In [5]:
df.groupBy("Origin").agg(
    round(avg("delay_target") * 100, 2)
    .alias("tasa_demora_porcentaje")
).orderBy(
    "tasa_demora_porcentaje",
    ascending=False
).show(10, truncate=False)

[Stage 13:===========================================>              (3 + 1) / 4]

+------+----------------------+
|Origin|tasa_demora_porcentaje|
+------+----------------------+
|DFW   |30.02                 |
|MCO   |27.7                  |
|LAS   |27.03                 |
|CLT   |26.16                 |
|DEN   |24.42                 |
|PHX   |24.01                 |
|ATL   |22.64                 |
|SEA   |21.32                 |
|LAX   |21.17                 |
+------+----------------------+



In [6]:
spark.stop()
print("Análisis ejecutado correctamente en Ruta A on-premise.")

Análisis ejecutado correctamente en Ruta A on-premise.
